In [1]:
import os
import json
import base64
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = "https://api.openai.com/v1"
WORKING_DIRECTORY_BASE_NAME = f"tmp_{datetime.now().strftime('%d.%m.%Y-%H:%M:%S')}"
IMAGE_BASE_PATH = "/u1/m2fetrat/GhCodes/visual-reasoning/Grounded-SAM/data/fsc-147/images_384_VarV2"
DATASET_PATH = "/u1/m2fetrat/GhCodes/visual-reasoning/Grounded-SAM/data/fsc-147/test.json"

os.mkdir(WORKING_DIRECTORY_BASE_NAME)

In [3]:
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
df = pd.read_json(DATASET_PATH)

In [4]:
def line_template_filler(id, question, image_bin):
    return {
        "custom_id": f"{id}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-2024-08-06",
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64.b64encode(image_bin).decode('utf-8')}"
                            }
                        },
                        {
                            "type": "text",
                            "text": question
                        }
                    ]
                }
            ],
            "temperature": 1,
            "max_tokens": 256,
            "top_p": 1,
            "frequency_penalty": 0,
            "presence_penalty": 0,
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "object_counter",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "count": {
                                "type": "integer"
                            }
                        },
                        "additionalProperties": False,
                        "required": [
                            "count"
                        ]
                    }
                }
            }
        }
    }

In [5]:
df

,id,image,object_of_interest,question,answer
0,0,2.jpg,sea shells,How many sea shells are there?,8
1,1,3.jpg,hot air balloons,How many hot air balloons are there?,11
2,2,4.jpg,hot air balloons,How many hot air balloons are there?,10
3,3,5.jpg,hot air balloons,How many hot air balloons are there?,113
4,4,6.jpg,hot air balloons,How many hot air balloons are there?,9
...,...,...,...,...,...
1185,1185,6918.jpg,nail polish,How many nail polish are there?,87
1186,1186,7500.jpg,sheep,How many sheep are there?,181
1187,1187,7047.jpg,sheep,How many sheep are there?,54
1188,1188,7412.jpg,sheep,How many sheep are there?,36


In [9]:
filename = "fsc147_remaining_400"
query = "800 <= index < 1200"

with open(f"{WORKING_DIRECTORY_BASE_NAME}/{filename}.jsonl", 'w') as request_file:
    for index, row in df.query(query).iterrows():
        with open(f"{IMAGE_BASE_PATH}/{row['image']}", 'rb') as f:
            image = f.read()
        # question = f"How many {row['object_of_interest']} are in the image?"
        jsonl = line_template_filler(id=row["id"], question=row['question'], image_bin=image)
        request_file.write(json.dumps(jsonl))
        request_file.write("\n")

batch_input_file = client.files.create(
  file=open(f"{WORKING_DIRECTORY_BASE_NAME}/{filename}.jsonl", "rb"),
  purpose="batch"
)

batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

Batch(id='batch_66fb0a2009d88190bab2fd4d44bf9103', completion_window='24h', created_at=1727728160, endpoint='/v1/chat/completions', input_file_id='file-p29RryPGdlyqEbDWGF7dzRcb', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1727814560, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

# Results

In [4]:
import glob
import json
import numpy as np
import pandas as pd

In [5]:
df = pd.read_json(DATASET_PATH)

In [15]:
llm_count = dict()
for file in glob.glob("/u1/m2fetrat/GhCodes/visual-reasoning/Grounded-SAM/tmp_30.09.2024-20:26:13/output0.jsonl"):
    with open(file) as f:
        for line in f:
            res = json.loads(line)
            llm_count[res['custom_id']] = json.loads(res['response']['body']['choices'][0]['message']['content'])['count']

In [20]:
df

,id,image,object_of_interest,question,answer,llm_count
0,0,2.jpg,sea shells,How many sea shells are there?,8,7
1,1,3.jpg,hot air balloons,How many hot air balloons are there?,11,12
2,2,4.jpg,hot air balloons,How many hot air balloons are there?,10,12
3,3,5.jpg,hot air balloons,How many hot air balloons are there?,113,89
4,4,6.jpg,hot air balloons,How many hot air balloons are there?,9,10
...,...,...,...,...,...,...
1185,1185,6918.jpg,nail polish,How many nail polish are there?,87,72
1186,1186,7500.jpg,sheep,How many sheep are there?,181,96
1187,1187,7047.jpg,sheep,How many sheep are there?,54,42
1188,1188,7412.jpg,sheep,How many sheep are there?,36,42


In [17]:
len(llm_count)

1190

In [18]:
df['llm_count'] = [None]*len(df)

In [19]:
for key, value in llm_count.items():
    df.loc[df['id'] == int(key), 'llm_count'] = value

In [21]:
df.to_csv("output0.csv")

In [41]:
def calculate_metrics(df):
    print("EA", (df['llm_count'].to_numpy() == df['answer'].to_numpy()).sum() / len(df))
    print("MAE", sum(abs(df['llm_count'].to_numpy() - df['answer'].to_numpy())) / len(df))
    print("RMSE", (np.sqrt(sum((df['llm_count'].to_numpy() - df['answer'].to_numpy())**2)/len(df))).item())

In [42]:
calculate_metrics(df)

EA 0.14369747899159663
MAE 17.43109243697479
RMSE 87.61360452280793
